# Module 3 — Introduction to Deep Learning

## Notebook 1: From Features to Neural Networks — Reference Solutions

This guided notebook introduces the main building blocks of a small artificial
neural network. You will calculate a forward pass, apply the ReLU activation
function, and train a small neural classifier on a two-dimensional dataset.

### Learning outcomes

By the end, you should be able to:

- identify inputs, weights, biases, layers, and activations;
- calculate a pre-activation value and apply ReLU;
- explain why a non-linear activation makes a neural network more expressive;
- distinguish a stack of linear layers from a network with a ReLU layer;
- interpret a decision-boundary plot for a linear model and a neural network.

Run the notebook from top to bottom. Then change the marked values and observe
how the outputs and plots change.
> **Organiser-controlled reference material:** This notebook contains complete implementations and model answers. It is released only through the Module 3 solution-release workflow.


In [ ]:
# These imports provide numerical arrays, plots, and small example datasets.
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# A fixed seed makes the generated dataset and train/test split reproducible.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Use a consistent plotting style throughout this notebook.
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

## 1. Inputs, weights, biases, and pre-activation

A neuron receives one or more input values. Each input is multiplied by a
**weight**. The weighted values are added together with a **bias**.

For inputs $$x_1, x_2$$, weights $$w_1, w_2$$, and bias $$b$$, the
pre-activation value is:

$$
z = x_1 w_1 + x_2 w_2 + b
$$

The pre-activation is often written as $$z$$. It is then passed to an
activation function.

In [ ]:
# A tiny example with two inputs and one neuron.
inputs = np.array([2.0, -1.0])
weights = np.array([0.8, -0.5])
bias = -0.2

# NumPy's dot product calculates x_1*w_1 + x_2*w_2.
pre_activation = np.dot(inputs, weights) + bias

print("inputs:         ", inputs)
print("weights:        ", weights)
print("bias:           ", bias)
print("pre-activation:", pre_activation)

## 2. ReLU: a simple non-linear activation function

The **Rectified Linear Unit** (ReLU) keeps positive values and replaces
negative values with zero:

$$
\operatorname{ReLU}(z) = \max(0, z)
$$

ReLU is popular because it is simple and gives a network a non-linear step.

In [ ]:
def relu(values):
    """Return the ReLU activation for one value or a NumPy array of values."""
    return np.maximum(0, values)

hidden_value = relu(pre_activation)

print("pre-activation:", pre_activation)
print("ReLU output:   ", hidden_value)

# Inspect ReLU over a range of possible pre-activation values.
z_values = np.linspace(-4, 4, 200)

plt.plot(z_values, relu(z_values), label="ReLU(z)", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Pre-activation z")
plt.ylabel("Activation")
plt.title("The ReLU activation function")
plt.legend()
plt.show()

### Try it yourself

Change one input, weight, or the bias in the next cell. Before running it,
predict whether the pre-activation and ReLU output will increase, decrease,
or become zero.

In [ ]:
# Change these values and run this cell again.
experiment_inputs = np.array([2.0, -1.0])
experiment_weights = np.array([0.8, -0.5])
experiment_bias = -0.2

experiment_pre_activation = (
    np.dot(experiment_inputs, experiment_weights) + experiment_bias
)
experiment_output = relu(experiment_pre_activation)

print("pre-activation:", experiment_pre_activation)
print("ReLU output:   ", experiment_output)

## 3. Why non-linear activations matter

A sequence of linear transformations is still only one linear transformation.
For example, if we apply two linear layers,

$$
h = XW_1 + b_1
$$

and

$$
y = hW_2 + b_2,
$$

then the whole calculation can be rewritten as a single linear transformation
of $$X$$. Adding ReLU between the layers changes this: the network can then
represent more complex shapes, including curved decision boundaries.

In [ ]:
# Create a few two-dimensional input examples.
X_demo = np.array([
    [-2.0, -1.0],
    [-1.0,  2.0],
    [ 0.0,  0.5],
    [ 1.0, -1.5],
    [ 2.0,  1.0],
])

# Define two small linear layers.
W1 = np.array([[1.0, -0.8], [0.5, 1.2]])
b1 = np.array([0.2, -0.4])
W2 = np.array([[0.7], [1.1]])
b2 = np.array([0.3])

# Two linear layers in sequence.
two_linear_layers = (X_demo @ W1 + b1) @ W2 + b2

# The same calculation written as one combined linear layer.
combined_weights = W1 @ W2
combined_bias = b1 @ W2 + b2
one_linear_layer = X_demo @ combined_weights + combined_bias

print("Two linear layers:\n", two_linear_layers.ravel())
print("\nOne combined linear layer:\n", one_linear_layer.ravel())
print(
    "\nAre they equal?",
    np.allclose(two_linear_layers, one_linear_layer),
)

# ReLU between the layers prevents this simple collapse into one linear layer.
relu_between_layers = relu(X_demo @ W1 + b1) @ W2 + b2
print("\nWith ReLU between the layers:\n", relu_between_layers.ravel())

## 4. A dataset with a curved pattern

The next dataset contains two interlocking curved groups. A straight line is
not sufficient to separate them well. This makes it useful for comparing a
linear classifier with a small neural network.

The dataset is bundled through scikit-learn and generated locally; no data is
downloaded from the internet.

In [ ]:
# make_moons creates two curved classes in two dimensions.
X, y = make_moons(
    n_samples=300,
    noise=0.22,
    random_state=RANDOM_SEED,
)

# Keep some data aside so that we can test generalisation later.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

plt.scatter(
    X[:, 0],
    X[:, 1],
    c=y,
    cmap="coolwarm",
    edgecolor="black",
    alpha=0.85,
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Two curved classes")
plt.show()

print("Training examples:", len(X_train))
print("Test examples:    ", len(X_test))

In [ ]:
def plot_decision_boundary(model, X_data, y_data, title):
    """Plot a fitted classifier's predicted class regions in two dimensions."""
    padding = 0.4
    x_min, x_max = X_data[:, 0].min() - padding, X_data[:, 0].max() + padding
    y_min, y_max = X_data[:, 1].min() - padding, X_data[:, 1].max() + padding

    x_grid, y_grid = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid_points = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    predicted_classes = model.predict(grid_points).reshape(x_grid.shape)

    plt.contourf(
        x_grid,
        y_grid,
        predicted_classes,
        alpha=0.25,
        cmap="coolwarm",
    )
    plt.scatter(
        X_data[:, 0],
        X_data[:, 1],
        c=y_data,
        cmap="coolwarm",
        edgecolor="black",
        alpha=0.85,
    )
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)

## 5. A linear model

Logistic regression is a useful baseline. It can create a linear decision
boundary, which means that its boundary is a straight line in this
two-dimensional feature space.

In [ ]:
# Train a deterministic linear classifier on the training data.
linear_model = LogisticRegression(random_state=RANDOM_SEED)
linear_model.fit(X_train, y_train)

linear_test_predictions = linear_model.predict(X_test)
linear_test_accuracy = accuracy_score(y_test, linear_test_predictions)

plot_decision_boundary(
    linear_model,
    X_test,
    y_test,
    f"Linear model decision boundary (test accuracy: {linear_test_accuracy:.2f})",
)
plt.show()

## 6. A small neural network with ReLU

The neural network below has one hidden layer with six neurons. Each hidden
neuron learns a different intermediate representation of the input. ReLU is
applied in the hidden layer, allowing the final decision boundary to bend.

`MLPClassifier` performs the repeated training updates for us. You do not need
to implement backpropagation in this workshop; the next notebook explains the
effect of training and gradients at a high level.

In [ ]:
# Train a small feed-forward neural network.
#
# hidden_layer_sizes=(6,) means one hidden layer containing six neurons.
# activation="relu" applies ReLU in that hidden layer.
# alpha is a small regularisation value.
neural_model = MLPClassifier(
    hidden_layer_sizes=(6,),
    activation="relu",
    solver="lbfgs",
    alpha=0.001,
    max_iter=1000,
    random_state=RANDOM_SEED,
)
neural_model.fit(X_train, y_train)

neural_test_predictions = neural_model.predict(X_test)
neural_test_accuracy = accuracy_score(y_test, neural_test_predictions)

plot_decision_boundary(
    neural_model,
    X_test,
    y_test,
    f"Neural-network decision boundary (test accuracy: {neural_test_accuracy:.2f})",
)
plt.show()

print(f"Linear-model test accuracy:     {linear_test_accuracy:.3f}")
print(f"Neural-network test accuracy:  {neural_test_accuracy:.3f}")

## 7. Looking at a learned hidden representation

The first layer transforms the original two input features into six hidden
features. The values below are the learned weights connecting the two input
features to the hidden neurons.

You do not need to interpret every individual number. The key idea is that the
network learns multiple weighted views of the same input, then combines them
after applying ReLU.

In [ ]:
# coefs_[0] contains the weights from the two input features to hidden neurons.
first_layer_weights = neural_model.coefs_[0]
first_layer_biases = neural_model.intercepts_[0]

print("First-layer weight matrix shape:", first_layer_weights.shape)
print("Rows correspond to input features; columns correspond to hidden neurons.")
print("\nFirst-layer weights:\n", np.round(first_layer_weights, 3))
print("\nFirst-layer biases:\n", np.round(first_layer_biases, 3))

# Calculate the ReLU-activated hidden representation for three training examples.
example_inputs = X_train[:3]
example_pre_activations = example_inputs @ first_layer_weights + first_layer_biases
example_hidden_features = relu(example_pre_activations)

print("\nThree input examples:\n", np.round(example_inputs, 3))
print("\nTheir ReLU hidden representations:\n", np.round(example_hidden_features, 3))

## 8. Summary and exit ticket

### Key ideas

- A neuron combines inputs, weights, and a bias to form a pre-activation.
- ReLU keeps positive values and maps negative values to zero.
- Stacked linear layers are equivalent to one linear layer.
- A non-linear activation such as ReLU lets a neural network model more
  complex decision regions.
- A hidden layer learns intermediate representations that help the output
  layer make a prediction.

### Exit ticket

Answer these questions in your own words:

1. What is the difference between a pre-activation and an activation?
2. Why is ReLU useful in a neural network?
3. Why did the neural network produce a curved decision boundary while the
   linear model produced a straight one?
4. Change the hidden-layer size from `(6,)` to `(2,)` and then `(20,)`.
   What changes do you observe in the decision boundary and test accuracy?

## Model answers for the exit ticket

1. A **pre-activation** is the weighted sum of inputs plus a bias. An
   **activation** is the result after applying an activation function, such as
   ReLU, to that pre-activation.

2. ReLU is useful because it introduces non-linearity. Without non-linear
   activations, several stacked linear layers are still equivalent to one
   linear transformation.

3. The linear model can only make a straight decision boundary in this
   two-feature space. The neural network combines several ReLU-activated hidden
   features, allowing it to construct a curved boundary.

4. With only two hidden neurons, the neural network may have insufficient
   capacity to represent the curved pattern well. With twenty hidden neurons,
   it can form a more detailed boundary, but a larger model can also become
   more likely to fit accidental detail in limited training data. The next
   notebook studies this trade-off.